# 05 — Kalman Filter (KF → EKF → UKF)

> **참고:** Udacity SFND_Unscented_Kalman_Filter
> **언어:** Python, C++
> **목표:** 선형 KF부터 비선형 UKF까지, Radar+LiDAR 융합

---

## 5-1. Bayes 필터 기초

모든 Kalman 필터의 기반.



```
사전 분포  p(x_k | z_{1:k-1})  =  ∫ p(x_k | x_{k-1}) · p(x_{k-1} | z_{1:k-1}) dx_{k-1}
                                           (운동 모델)        (이전 사후 분포)

사후 분포  p(x_k | z_{1:k})   ∝  p(z_k | x_k) · p(x_k | z_{1:k-1})
                                      (관측 모델)     (사전 분포)
```




상태 = Gaussian이면 → Kalman Filter로 정확한 Bayes update 가능.

---

## 5-2. 선형 Kalman Filter

**상태 모델 (선형):**


```
x_k = F·x_{k-1} + B·u_k + w_k     (w_k ~ N(0, Q))
z_k = H·x_k + v_k                  (v_k ~ N(0, R))
```


In [ ]:
import numpy as np

class KalmanFilter:
    def __init__(self, F, H, Q, R, x0, P0):
        self.F = F   # 상태 전이 행렬
        self.H = H   # 관측 행렬
        self.Q = Q   # 프로세스 노이즈 공분산
        self.R = R   # 관측 노이즈 공분산
        self.x = x0  # 초기 상태
        self.P = P0  # 초기 공분산

    def predict(self):
        self.x = self.F @ self.x
        self.P = self.F @ self.P @ self.F.T + self.Q

    def update(self, z: np.ndarray):
        S = self.H @ self.P @ self.H.T + self.R        # 혁신 공분산
        K = self.P @ self.H.T @ np.linalg.inv(S)       # Kalman Gain
        y = z - self.H @ self.x                         # 혁신 (innovation)
        self.x = self.x + K @ y
        self.P = (np.eye(len(self.x)) - K @ self.H) @ self.P

# 2D 등속 운동 모델 예시
dt = 0.1
F = np.array([[1, 0, dt, 0],
              [0, 1, 0, dt],
              [0, 0, 1,  0],
              [0, 0, 0,  1]])
H = np.array([[1, 0, 0, 0],
              [0, 1, 0, 0]])
Q = np.eye(4) * 0.01
R = np.eye(2) * 0.5
x0 = np.array([0., 0., 1., 0.5])
P0 = np.eye(4) * 1.0

kf = KalmanFilter(F, H, Q, R, x0, P0)
for z in measurements:
    kf.predict()
    kf.update(z)
    print(f"State: {kf.x.round(3)}")




---

## 5-3. Extended Kalman Filter (EKF)

비선형 시스템 → Jacobian으로 선형화.



```
x_k = f(x_{k-1}) + w_k        (비선형 전이)
z_k = h(x_k) + v_k            (비선형 관측)

선형화: F ≈ ∂f/∂x|_{x̂}   (Jacobian)
        H ≈ ∂h/∂x|_{x̂}   (Jacobian)
```




**Radar 관측 모델 (극좌표):**


```
z = h(x) = [ρ, φ, ρ̇]
  ρ  = √(px² + py²)
  φ  = atan2(py, px)
  ρ̇  = (px·vx + py·vy) / ρ
```


In [ ]:
class EKF:
    def __init__(self, Q, R_lidar, R_radar, x0, P0):
        self.Q = Q
        self.R_lidar = R_lidar   # (2, 2)
        self.R_radar = R_radar   # (3, 3)
        self.x = x0   # [px, py, vx, vy]
        self.P = P0

    def _F(self, dt):
        return np.array([[1, 0, dt, 0],
                         [0, 1, 0, dt],
                         [0, 0, 1,  0],
                         [0, 0, 0,  1]])

    def _Q(self, dt, sigma_ax=9.0, sigma_ay=9.0):
        """프로세스 노이즈 (가속도 불확실성)"""
        dt2 = dt**2; dt3 = dt**3; dt4 = dt**4
        return np.array([
            [dt4/4*sigma_ax, 0, dt3/2*sigma_ax, 0],
            [0, dt4/4*sigma_ay, 0, dt3/2*sigma_ay],
            [dt3/2*sigma_ax, 0, dt2*sigma_ax, 0],
            [0, dt3/2*sigma_ay, 0, dt2*sigma_ay]
        ])

    def predict(self, dt):
        F = self._F(dt)
        self.x = F @ self.x
        self.P = F @ self.P @ F.T + self._Q(dt)

    def _radar_h(self, x):
        px, py, vx, vy = x
        rho = np.sqrt(px**2 + py**2)
        if rho < 1e-6: rho = 1e-6
        phi = np.arctan2(py, px)
        rho_dot = (px*vx + py*vy) / rho
        return np.array([rho, phi, rho_dot])

    def _radar_H(self, x):
        """Radar h(x)의 Jacobian"""
        px, py, vx, vy = x
        rho2 = px**2 + py**2
        rho  = np.sqrt(rho2)
        rho3 = rho2 * rho
        return np.array([
            [px/rho,  py/rho,  0, 0],
            [-py/rho2, px/rho2, 0, 0],
            [py*(vx*py-vy*px)/rho3, px*(vy*px-vx*py)/rho3, px/rho, py/rho]
        ])

    def update_lidar(self, z):
        H = np.array([[1,0,0,0],[0,1,0,0]])
        y = z - H @ self.x
        S = H @ self.P @ H.T + self.R_lidar
        K = self.P @ H.T @ np.linalg.inv(S)
        self.x = self.x + K @ y
        self.P = (np.eye(4) - K @ H) @ self.P

    def update_radar(self, z):
        H = self._radar_H(self.x)
        y = z - self._radar_h(self.x)
        # 각도 정규화 [-π, π]
        y[1] = np.arctan2(np.sin(y[1]), np.cos(y[1]))
        S = H @ self.P @ H.T + self.R_radar
        K = self.P @ H.T @ np.linalg.inv(S)
        self.x = self.x + K @ y
        self.P = (np.eye(4) - K @ H) @ self.P




---

## 5-4. Unscented Kalman Filter (UKF)

Jacobian 불필요. Sigma point 전파로 비선형 변환 처리.

### CTRV 운동 모델 (Constant Turn Rate and Velocity)



```
상태: x = [px, py, v, ψ, ψ̇]   (위치, 속도 크기, 방위각, 각속도)

f(x, dt):
  if |ψ̇| > 1e-5:              (곡선 운동)
    px' = px + v/ψ̇ · (sin(ψ + ψ̇dt) - sinψ)
    py' = py + v/ψ̇ · (-cos(ψ + ψ̇dt) + cosψ)
  else:                         (직선 운동)
    px' = px + v·cos(ψ)·dt
    py' = py + v·sin(ψ)·dt
  v'  = v
  ψ'  = ψ + ψ̇·dt
  ψ̇' = ψ̇
```


In [ ]:
class UKF:
    def __init__(self, n_x=5, alpha=0.001, beta=2, kappa=0):
        self.n_x = n_x
        self.n_aug = n_x + 2    # 프로세스 노이즈 2개 (a_x, ψ̈)
        self.alpha = alpha
        self.beta  = beta
        self.kappa = kappa

        lam = alpha**2 * (n_x + kappa) - n_x
        self.lam = lam
        n = n_x
        # Sigma point 가중치
        self.Wm = np.array([lam/(n+lam)] +
                            [0.5/(n+lam)] * (2*n))
        self.Wc = np.array([lam/(n+lam) + (1-alpha**2+beta)] +
                            [0.5/(n+lam)] * (2*n))

        self.x = np.zeros(n_x)
        self.P = np.eye(n_x)
        self.Q = np.diag([0.0, 0.0, 0.09, 0.0, 0.009])  # 프로세스 노이즈
        self.R_radar = np.diag([0.09, 0.0009, 0.09])     # Radar 관측 노이즈

    def _sigma_points(self):
        n = self.n_x
        lam = self.lam
        L = np.linalg.cholesky((n + lam) * self.P)
        sigma = np.zeros((2*n+1, n))
        sigma[0] = self.x
        for i in range(n):
            sigma[i+1]   = self.x + L[:, i]
            sigma[i+1+n] = self.x - L[:, i]
        return sigma

    def _ctrv(self, x: np.ndarray, dt: float, noise: np.ndarray) -> np.ndarray:
        px, py, v, psi, psi_dot = x
        nu_a, nu_psi_ddot = noise
        if abs(psi_dot) > 1e-5:
            px_new = px + v/psi_dot * (np.sin(psi+psi_dot*dt) - np.sin(psi))
            py_new = py + v/psi_dot * (-np.cos(psi+psi_dot*dt) + np.cos(psi))
        else:
            px_new = px + v*np.cos(psi)*dt
            py_new = py + v*np.sin(psi)*dt
        v_new   = v   + nu_a*dt
        psi_new = psi + psi_dot*dt + 0.5*nu_psi_ddot*dt**2
        psi_dot_new = psi_dot + nu_psi_ddot*dt
        return np.array([px_new, py_new, v_new, psi_new, psi_dot_new])

    def predict(self, dt: float):
        sigma = self._sigma_points()     # (2n+1, n)
        sigma_pred = np.array([self._ctrv(s, dt, np.zeros(2)) for s in sigma])

        # 예측 평균
        self.x = self.Wm @ sigma_pred
        # 예측 공분산
        diff = sigma_pred - self.x
        self.P = sum(self.Wc[i] * np.outer(diff[i], diff[i])
                     for i in range(2*self.n_x+1)) + self.Q

    def update_radar(self, z: np.ndarray):
        sigma = self._sigma_points()
        # 측정 공간으로 변환
        def h(x): return np.array([
            np.sqrt(x[0]**2+x[1]**2),
            np.arctan2(x[1], x[0]),
            (x[0]*x[2]*np.cos(x[3]) + x[1]*x[2]*np.sin(x[3])) /
            max(np.sqrt(x[0]**2+x[1]**2), 1e-6)
        ])
        Z_sigma = np.array([h(s) for s in sigma])
        z_pred = self.Wm @ Z_sigma
        # 혁신 공분산
        dz = Z_sigma - z_pred
        dz[:, 1] = np.arctan2(np.sin(dz[:, 1]), np.cos(dz[:, 1]))
        S = sum(self.Wc[i] * np.outer(dz[i], dz[i]) for i in range(len(Z_sigma))) + self.R_radar
        # 교차 공분산
        dx = sigma - self.x
        Tc = sum(self.Wc[i] * np.outer(dx[i], dz[i]) for i in range(len(sigma)))
        K = Tc @ np.linalg.inv(S)
        y = z - z_pred
        y[1] = np.arctan2(np.sin(y[1]), np.cos(y[1]))
        self.x = self.x + K @ y
        self.P = self.P - K @ S @ K.T




---

## 5-5. Normalized Innovation Squared (NIS) — 일관성 검증



In [ ]:
def nis_radar(z, z_pred, S):
    """
    NIS = ε = (z - z_pred)ᵀ S⁻¹ (z - z_pred)
    이론적으로 chi-squared 분포 따름 (자유도 = 관측 차원)
    Radar 3D: 95% 범위 → [0.352, 7.815]
    """
    y = z - z_pred
    return float(y.T @ np.linalg.inv(S) @ y)

# NIS 시퀀스 플롯 → 95% 범위 내에 있으면 노이즈 파라미터 적절
nis_values = []
for z in radar_measurements:
    ukf.predict(dt)
    # z_pred, S는 update 전에 계산
    nis_values.append(nis_radar(z, z_pred, S))
    ukf.update_radar(z)

import matplotlib.pyplot as plt
plt.plot(nis_values, label='NIS')
plt.axhline(7.815, color='r', linestyle='--', label='95% upper (chi2, df=3)')
plt.axhline(0.352, color='r', linestyle='--', label='95% lower')
plt.legend(); plt.title('NIS Consistency Check')




---

## 5-6. 성능 비교: KF vs EKF vs UKF

| 항목 | Linear KF | EKF | UKF |
|------|----------|-----|-----|
| 모델 | 선형 | 비선형 (Jacobian) | 비선형 (Sigma points) |
| 정확도 | 정확 (선형) | 1차 근사 | 2차 이상 근사 |
| Jacobian 필요 | 불필요 | 필요 | 불필요 |
| 계산량 | O(n³) | O(n³) + Jac 계산 | O(n³) (2n+1 sigma) |
| 추천 상황 | 등속 직선 | Radar 관측 | CTRV, 고기동 타깃 |

---

## 핵심 개념 정리

| 개념 | 핵심 |
|------|------|
| Kalman Gain K | 예측 불확실성 vs 관측 노이즈 trade-off |
| 혁신 y | z - H·x̂, 예측과 실제 측정의 차이 |
| NIS | 필터 일관성 검증, chi² 분포와 비교 |
| CTRV | 곡선 운동에 적합한 비선형 운동 모델 |
| Sigma points | 비선형 변환의 평균/공분산을 deterministic하게 근사 |

---

## 다음 단계

→ **06_sensor_fusion.md** : 데이터 연관, 다중 센서 3D 추적
